# Top-down monthly and daily backtest comparison

**February 2020–April 2026: 75 months / 1,569 sessions.** All total returns compound over the full 75 months. Annualized CAGR appears separately. The four Paper15 monthly-label models retain their exact saved RET selections.

The first table gives the **separate monthly-engine result**, daily same-close control, daily next-open gross, daily next-open net and the asset-stop alternative. Top-down monthly means realized monthly portfolio returns, including approved cash/share merger entitlements. RET is the expected-payoff ranking score used to select stocks.

Run All reads verified aggregate JSON, bounded proof artifacts and saved chart images. It starts no fit, replay, download or source-panel read. [Method and full results](../docs/WALKFORWARD_EXTENSION_BACKTEST_20261027.md).

In [ ]:
from pathlib import Path
import json, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
local = ROOT / "configs/full_period_comparison.local.json"
settings = json.loads(local.read_text()) if local.exists() else {}
reporting_packages = Path(settings.get("reporting_packages", ROOT / ".reporting"))
if reporting_packages.is_dir():
    sys.path.insert(0, str(reporting_packages))
REPORT_DIRECTORY = Path(settings.get("output_directory", ROOT / "outputs/full_period_comparison_report/20261027-v1"))
from dmv2.full_period_comparison_reporting import read_full_period_comparison
report, saved = None, None
try:
    report = read_full_period_comparison(REPORT_DIRECTORY)
except FileNotFoundError:
    print("Private verified comparison is unavailable. No compute started.")
else:
    saved = report["daily"]
    print(report["status"], "|", report["period"], "|", report["months"], "months /", report["sessions"], "sessions")
    print("48 daily paths and 16 monthly references verified; zero withheld. No refits or new backtests for this report.")


## Cumulative compounded total returns over 75 months

The monthly and daily same-close columns come from separate saved backtest tables. Both use the same close-to-close clock, quote view and legal-entitlement convention. Daily next-open execution changes the timing. Net columns charge **10 bp per one-way trade**; the last adds the unchanged **10% per-asset stop**.

All columns use the same saved names, sides and weights from the monthly-label models. Earlier models trained on executable daily targets and expanded-RF targets remain separate experiments.

In [ ]:
if report:
    import pandas as pd
    from IPython.display import display, Image
    results = pd.DataFrame(saved["comparison"])
    columns = {
        "monthly_gross": "Top-down monthly gross total",
        "daily_same_close_gross": "Daily same-close gross total",
        "daily_next_open_gross": "Daily next-open gross total",
        "daily_next_open_net10": "Daily next-open net total, 10 bp",
        "daily_next_open_net10_stop": "Daily next-open net total, 10 bp + asset stops",
    }
    table = pd.DataFrame(report["comparison"]).set_index("model")[list(columns)].rename(columns=columns)
    display(table.style.format("{:+.2%}", na_rep="WITHHELD").set_caption("Cumulative compounded TOTAL RETURN · February 2020–April 2026 · 75 months"))
    display(Image(filename=saved["figures"]["equity_comparison"]))


## Annualized and daily risk measures

CAGR uses the frozen 252/1,569 daily convention; Sharpe uses daily returns and zero cash rate. Maximum drawdown is measured from daily closes, including the starting NAV. The chart shows daily drawdown for net returns with asset stops. Stops can worsen returns; gap fills and idealized threshold fills are assumptions, not guaranteed executable prices.

In [ ]:
if saved:
    risk = results.loc[results.stage.isin(["execution_open_net", "execution_open_net_stop"])].set_index(["arm", "stage"])[["complete_path", "total_return", "cagr", "daily_sharpe", "daily_volatility", "daily_maximum_drawdown", "stop_count"]]
    display(risk.style.format({"total_return":"{:+.2%}", "cagr":"{:+.2%}", "daily_sharpe":"{:.2f}", "daily_volatility":"{:.2%}", "daily_maximum_drawdown":"{:.2%}"}, na_rep="WITHHELD"))
    display(Image(filename=saved["figures"]["daily_drawdown"]))

## Complete cost sensitivity and continuous annual slices

Costs are 0, 5, 10 and 25 bp per one-way voluntary trade. Zero borrow and financing charges remain an explicit limitation. **2020 has 11 months; 2026 has January–April only.** Annual rows are slices of the continuous portfolio, not independent refits or resets. U.S. Steel’s June 2025 signed cash entitlement affects realized holdings only; labels, RET and saved predictions remain frozen.

In [ ]:
if saved:
    display(results.set_index(["arm", "stage"])[["complete_path", "cost_bps", "asset_stop_fraction", "total_return", "cagr", "daily_sharpe", "daily_maximum_drawdown", "stop_count", "first_failure_date"]])
    annual = pd.DataFrame(saved["annual"])
    display(annual.loc[annual.stage.isin(["execution_open_net", "execution_open_net_stop"])].set_index(["arm", "stage", "year"]).style.format({"total_return":"{:+.2%}"}))
    print("Matching monthly/daily same-close controls agree to numerical precision over all 75 months.")
    for limitation in saved["limits"]:
        print("•", limitation)